# Dataset Scaling Experiment: Learning Curves

This notebook measures how recommendation metrics evolve as the training set grows.
The training partition is consumed in 7 cumulative fractions (1/7, 2/7, ..., 7/7) while the test set is held fixed.

In [1]:
%cd ..

/home/jovyan/project/film-recommendation


In [2]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_processing.data_loader import MovieLensDataLoader
from src.models.content_based import ContentBasedRecommender, ContentBasedConfig
from src.models.collaborative_filtering import CollaborativeFiltering
from src.models.hybrid import HybridRecommender
from src.models.cascading_hybrid import CascadingHybridRecommender
from src.models.popular_baseline_model import PopularityBaseline
from src.evaluation.evaluator import RecommendationEvaluator
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)
OUTPUT_DIR = 'scaling_experiment'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load Data

In [3]:
import asyncio

loader = MovieLensDataLoader('ml-latest')
data_dict = loader.load_data()
ratings_df = data_dict['ratings']

# ← ГЛАВНЫЙ FIX: вызвать async-загрузку TMDB (берёт из кеша если он есть)
await loader.letterboxd_data_async(max_concurrent_requests=500)

# Базовый MovieLens (movieId, title, genres)
base_movies = data_dict['movies'].copy()

# Genre one-hot
genre_features = loader.preprocess_movies()  # индекс совпадает с base_movies

# TMDB-обогащение
if loader.movie_data:
    tmdb_df = pd.DataFrame(loader.movie_data)
    # убираем title из TMDB чтобы не дублировать
    tmdb_df = tmdb_df.drop(columns=['title'], errors='ignore')
    movies_df = base_movies.merge(tmdb_df, on='movieId', how='left')
else:
    movies_df = base_movies.copy()

# Сбрасываем индекс перед concat с genre_features
movies_df = movies_df.reset_index(drop=True)
genre_features = genre_features.reset_index(drop=True)
movies_df = pd.concat([movies_df, genre_features], axis=1)

movies_df = movies_df.dropna(subset=['movieId']).reset_index(drop=True)
print(f"Movies: {len(movies_df)}")
print(f"Ratings: {len(ratings_df)}")
print(f"Users: {ratings_df['userId'].nunique()}")

INFO:src.data_processing.data_loader:Loading MovieLens dataset...
INFO:src.data_processing.data_loader:Loading existing data from cache: data/processed/movie_metadata.parquet
INFO:src.data_processing.data_loader:Found 1180 missing movies with valid TMDB IDs. Fetching...
INFO:src.data_processing.data_loader:DOWNLOAD SUMMARY: Requested: 1180 | Saved: 0 | Failed: 1180


Movies: 86537
Ratings: 33832162
Users: 330975


## 2. Temporal Split (Fixed Test Set)

In [4]:
ratings_df = ratings_df.sort_values('timestamp').reset_index(drop=True)
split_idx = int(len(ratings_df) * 0.8)
full_train_df = ratings_df.iloc[:split_idx].copy()
test_df = ratings_df.iloc[split_idx:].copy()
print(f"Full train: {len(full_train_df)} ratings")
print(f"Test (fixed): {len(test_df)} ratings")
print(f"Test users: {test_df['userId'].nunique()}")

Full train: 27065729 ratings
Test (fixed): 6766433 ratings
Test users: 58661


## 3. Define Scale Fractions

In [5]:
N_SCALES = 7
EVAL_SAMPLE_SIZE = 5000
K_VALUES = [5, 10, 20]
fractions = [i / N_SCALES for i in range(1, N_SCALES + 1)]
print(f"Fractions: {fractions}")
for f in fractions:
    print(f"  {f:.3f} -> {int(len(full_train_df) * f)} ratings")

Fractions: [0.14285714285714285, 0.2857142857142857, 0.42857142857142855, 0.5714285714285714, 0.7142857142857143, 0.8571428571428571, 1.0]
  0.143 -> 3866532 ratings
  0.286 -> 7733065 ratings
  0.429 -> 11599598 ratings
  0.571 -> 15466130 ratings
  0.714 -> 19332663 ratings
  0.857 -> 23199196 ratings
  1.000 -> 27065729 ratings


## 4. Run Scaling Experiment

In [ ]:
all_records = []

for frac in fractions:
    n_ratings = int(len(full_train_df) * frac)
    train_df = full_train_df.iloc[:n_ratings].copy()
    print(f"\n{'='*70}")
    print(f"SCALE {frac:.3f} | Train: {len(train_df)} | Users: {train_df['userId'].nunique()}")
    print(f"{'='*70}")

    models = {}

    try:
        cb_config = ContentBasedConfig(
            main_actor_weight=0.3, director_weight=0.3, cast_weight=0.3,
            keywords_weight=0.6, numerical_weight=0.1,
            similarity_threshold=0.15, top_k_default=20
        )
        cb_model = ContentBasedRecommender(config=cb_config)
        cb_model.fit(movies_df=movies_df, ratings_df=train_df)
        models['Content-Based'] = cb_model
        print("[OK] Content-Based")
    except Exception as e:
        print(f"[FAIL] Content-Based: {e}")

    try:
        cf_model = CollaborativeFiltering(k_components=110, random_state=42)
        cf_model.fit(df_ratings=train_df)
        models['Collaborative'] = cf_model
        print("[OK] Collaborative")
    except Exception as e:
        print(f"[FAIL] Collaborative: {e}")

    try:
        if 'Collaborative' in models and 'Content-Based' in models:
            hybrid = HybridRecommender(
                cf_model=models['Collaborative'],
                cb_model=models['Content-Based'],
                alpha=0.8
            )
            hybrid.fitted(
                cf_model=models['Collaborative'],
                cb_model=models['Content-Based'],
                movies_df=movies_df,
                ratings_df=train_df
            )
            models['Hybrid'] = hybrid
            print("[OK] Hybrid")
        else:
            print("[SKIP] Hybrid: requires CF + CB")
    except Exception as e:
        print(f"[FAIL] Hybrid: {e}")

    try:
        pop = PopularityBaseline()
        pop.fit(train_df)
        models['Popularity'] = pop
        print("[OK] Popularity")
    except Exception as e:
        print(f"[FAIL] Popularity: {e}")

    try:
        if 'Collaborative' in models and 'Content-Based' in models:
            casc = CascadingHybridRecommender(
                primary_model=models['Collaborative'],
                secondary_model=models['Content-Based'],
                primary_k=50
            )
            casc.fitted(
                primary_model=models['Collaborative'],
                secondary_model=models['Content-Based']
            )
            models['Cascading Hybrid'] = casc
            print("[OK] Cascading Hybrid")
        else:
            print("[SKIP] Cascading Hybrid: requires CF + CB")
    except Exception as e:
        print(f"[FAIL] Cascading Hybrid: {e}")

    if not models:
        print("No models trained, skipping evaluation")
        continue

    try:
        evaluator = RecommendationEvaluator(
            models=models,
            train_df=train_df,
            test_df=test_df,
            relevance_threshold=4.0,
            user_sample_size=EVAL_SAMPLE_SIZE,
            random_state=42
        )
        results_df = evaluator.evaluate_all_models(k_values=K_VALUES, max_recommendations=20)
        results_df['fraction'] = frac
        results_df['n_train_ratings'] = len(train_df)
        all_records.append(results_df)
        print(f"Evaluated {len(models)} models")
    except Exception as e:
        print(f"Evaluation failed: {e}")

    del models
    gc.collect()

scaling_results = pd.concat(all_records, ignore_index=True) if all_records else pd.DataFrame()
scaling_results.to_parquet(f'{OUTPUT_DIR}/results.parquet', index=False)
print(f"\nTotal records: {len(scaling_results)}")

2026-06-30 21:50:24.394 | INFO     | src.models.content_based:fit:616 - Starting model fitting
2026-06-30 21:50:24.400 | INFO     | src.models.content_based:fit:623 - Cleaning text columns



SCALE 0.143 | Train: 3866532 | Users: 67071


2026-06-30 21:50:24.646 | INFO     | src.models.content_based:fit:627 - Building cast and keyword strings
2026-06-30 21:50:24.646 | INFO     | src.models.content_based:fit:629 - Processing cast entries


Building cast strings:   0%|          | 0/86537 [00:00<?, ?it/s]

Building keyword strings:   0%|          | 0/86537 [00:00<?, ?it/s]

...

Collaborative:   0%|          | 0/3 [00:00<?, ?it/s]

2026-07-01 00:34:43.524 | INFO     | src.evaluation.evaluator:evaluate_model:447 - 'Collaborative' done in 0.3s
2026-07-01 00:34:43.525 | INFO     | src.evaluation.evaluator:evaluate_model:226 - Evaluating 'Hybrid'


Hybrid:   0%|          | 0/3 [00:00<?, ?it/s]

2026-07-01 00:34:43.530 | WARNING  | src.evaluation.evaluator:evaluate_model:297 - Batch error for Hybrid: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all(). Falling back to single mode.


Hybrid:   0%|          | 0/5000 [00:00<?, ?it/s]

## 5. Learning Curves

In [ ]:
if scaling_results.empty:
    raise RuntimeError("No results to plot")

metric_cols = ['precision', 'recall', 'ndcg', 'map', 'mrr', 'novelty', 'coverage']
available_metrics = [m for m in metric_cols if m in scaling_results.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, metric in enumerate(available_metrics[:6]):
    ax = axes[idx]
    for model_name in scaling_results['model'].unique():
        subset = scaling_results[
            (scaling_results['model'] == model_name) & (scaling_results['k'] == 10)
        ].sort_values('n_train_ratings')
        ax.plot(subset['n_train_ratings'], subset[metric], marker='o', label=model_name)
    ax.set_title(f'{metric.upper()}@10 vs Dataset Size')
    ax.set_xlabel('Training Ratings (log scale)')
    ax.set_ylabel(metric.upper())
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

for idx in range(len(available_metrics[:6]), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Final Comparison at Full Scale

In [ ]:
full_scale = scaling_results[scaling_results['fraction'] == 1.0]
if not full_scale.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for idx, metric in enumerate(['precision', 'recall', 'ndcg']):
        ax = axes[idx]
        data = full_scale[full_scale['k'] == 10].sort_values(metric, ascending=True)
        colors = sns.color_palette('viridis', len(data))
        ax.barh(data['model'], data[metric], color=colors)
        ax.set_title(f'{metric.upper()}@10 (Full Data)')
        ax.set_xlabel(metric.upper())
        for i, v in enumerate(data[metric]):
            ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/final_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Scaling Summary Table

In [ ]:
if not scaling_results.empty:
    summary = scaling_results[scaling_results['k'] == 10].pivot_table(
        index=['fraction', 'n_train_ratings'],
        columns='model',
        values=['precision', 'recall', 'ndcg']
    )
    print(summary.to_string())
    summary.to_csv(f'{OUTPUT_DIR}/summary.csv')